#Parallel Image Processing
### Follow these steps to set up parallel image processing:

1. Create Python files

 * parallel_image_processing.py

 * threaded_image_processing.py

2. Copy the code

 * Use the code provided at the bottom of this document.

3. Upload both files

 * Ensure both .py files are ready for execution.

In [6]:
# Test with more expensive operation (larger Gaussian blur)
!python parallel_image_processing.py \
  --input 743194.jpg \
  --output out_gaussian.jpg \
  --op gaussian \
  --ksize 21 \
  --tiles 4x4 \
  --halo 15 \
  --workers 4

# Test with even more tiles
!python parallel_image_processing.py \
  --input 743194.jpg \
  --output out_canny_8x8.jpg \
  --op canny \
  --tiles 8x8 \
  --halo 8 \
  --workers 8

# Test median filter (computationally expensive)
!python parallel_image_processing.py \
  --input 743194.jpg \
  --output out_median.jpg \
  --op median \
  --ksize 11 \
  --tiles 4x4 \
  --halo 8 \
  --workers 4

python3: can't open file '/content/parallel_image_processing.py': [Errno 2] No such file or directory
python3: can't open file '/content/parallel_image_processing.py': [Errno 2] No such file or directory
python3: can't open file '/content/parallel_image_processing.py': [Errno 2] No such file or directory


In [8]:
# Test threaded version with median filter (your best case)
!python threaded_image_processing.py \
  --input 743194.jpg \
  --output out_threaded_median.jpg \
  --op median \
  --ksize 11 \
  --tiles 4x4 \
  --workers 4

# Test with Canny
!python threaded_image_processing.py \
  --input 743194.jpg \
  --output out_threaded_canny.jpg \
  --op canny \
  --tiles 4x4 \
  --workers 4

# Test with more workers
!python threaded_image_processing.py \
  --input 743194.jpg \
  --output out_threaded_gaussian.jpg \
  --op gaussian \
  --ksize 21 \
  --tiles 6x6 \
  --workers 8

Input image shape: (2000, 3000, 3)
Sequential output shape: (2000, 3000, 3)
Threaded parallel output shape: (2000, 3000, 3)
Operation         : median
Image             : 743194.jpg -> out_threaded_median.jpg
Tiles (YxX)       : 4x4 | Halo: 8 | Workers: 4
Sequential time   : 661.99 ms
Threaded time     : 826.78 ms
Speedup           : 0.80x
Threading overhead: +24.9%
Input image shape: (2000, 3000, 3)
Sequential output shape: (2000, 3000)
Threaded parallel output shape: (2000, 3000)
Operation         : canny
Image             : 743194.jpg -> out_threaded_canny.jpg
Tiles (YxX)       : 4x4 | Halo: 8 | Workers: 4
Sequential time   : 67.36 ms
Threaded time     : 65.12 ms
Speedup           : 1.03x
Threading overhead: -3.3%
Input image shape: (2000, 3000, 3)
Sequential output shape: (2000, 3000, 3)
Threaded parallel output shape: (2000, 3000, 3)
Operation         : gaussian
Image             : 743194.jpg -> out_threaded_gaussian.jpg
Tiles (YxX)       : 6x6 | Halo: 8 | Workers: 8
Sequential ti

In [5]:
# Try Canny with more tiles (it's working!)
!python "/content/drive/MyDrive/Colab Notebooks/threaded_image_processing.py" \
  --input "/content/drive/MyDrive/Colab Notebooks/743194.jpg" \
  --output out_canny_8x8.jpg \
  --op canny \
  --tiles 8x8 \
  --workers 8

# Try Sobel (similar to Canny)
!python "/content/drive/MyDrive/Colab Notebooks/threaded_image_processing.py" \
  --input "/content/drive/MyDrive/Colab Notebooks/743194.jpg" \
  --output out_sobel.jpg \
  --op sobel \
  --tiles 4x4 \
  --workers 4

# Try Canny with different worker counts
!python "/content/drive/MyDrive/Colab Notebooks/threaded_image_processing.py" \
  --input "/content/drive/MyDrive/Colab Notebooks/743194.jpg" \
  --output out_canny_2workers.jpg \
  --op canny \
  --tiles 4x4 \
  --workers 2

# Try sharpening filter
!python "/content/drive/MyDrive/Colab Notebooks/threaded_image_processing.py" \
  --input "/content/drive/MyDrive/Colab Notebooks/743194.jpg" \
  --output out_sharpen.jpg \
  --op sharpen \
  --tiles 4x4 \
  --workers 4

Input image shape: (2000, 3000, 3)
Sequential output shape: (2000, 3000)
Threaded parallel output shape: (2000, 3000)
Operation         : canny
Image             : /content/drive/MyDrive/Colab Notebooks/743194.jpg -> out_canny_8x8.jpg
Tiles (YxX)       : 8x8 | Halo: 8 | Workers: 8
Sequential time   : 100.88 ms
Threaded time     : 98.44 ms
Speedup           : 1.02x
Threading overhead: -2.4%
Input image shape: (2000, 3000, 3)
Sequential output shape: (2000, 3000)
Threaded parallel output shape: (2000, 3000)
Operation         : sobel
Image             : /content/drive/MyDrive/Colab Notebooks/743194.jpg -> out_sobel.jpg
Tiles (YxX)       : 4x4 | Halo: 8 | Workers: 4
Sequential time   : 88.01 ms
Threaded time     : 66.02 ms
Speedup           : 1.33x
Threading overhead: -25.0%
Input image shape: (2000, 3000, 3)
Sequential output shape: (2000, 3000)
Threaded parallel output shape: (2000, 3000)
Operation         : canny
Image             : /content/drive/MyDrive/Colab Notebooks/743194.jpg -> o

In [ ]:
#!/usr/bin/env python3
"""
Parallel Image Processing (CPU, tile-based) - FIXED VERSION

Features
- Runs classic filters (gaussian blur, median, sobel, canny, sharpen) in parallel
- Splits image into tiles with halo/overlap so edges are correct
- Compares sequential vs parallel runtime
- Saves output image

Usage
------
python parallel_image_processing.py \
  --input input.jpg \
  --output out.jpg \
  --op canny \
  --tiles 2x4 \
  --grayscale

Supported ops: gaussian, median, sobel, canny, sharpen

Notes
-----
- Uses ProcessPoolExecutor (multi-core CPU). No GPU dependencies required.
- For very small images, parallel may be slower due to overhead.
- Tune --tiles and --halo for best results.
"""

from __future__ import annotations
import argparse
import math
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from dataclasses import dataclass
from typing import List, Tuple
import time

import cv2
import numpy as np


# ---------------------------
# Utility dataclasses
# ---------------------------
@dataclass
class Tile:
    x0: int
    y0: int
    x1: int
    y1: int
    # crop area inside the halo-expanded ROI to keep
    crop: Tuple[int, int, int, int]  # (cx0, cy0, cx1, cy1) relative to ROI


# ---------------------------
# Operations
# ---------------------------

def apply_op(img: np.ndarray, op: str, **kwargs) -> np.ndarray:
    op = op.lower()
    if op == 'gaussian':
        k = int(kwargs.get('ksize', 5))
        k = k if k % 2 == 1 else k + 1
        sigma = float(kwargs.get('sigma', 1.0))
        return cv2.GaussianBlur(img, (k, k), sigma)
    elif op == 'median':
        k = int(kwargs.get('ksize', 5))
        k = k if k % 2 == 1 else k + 1
        return cv2.medianBlur(img, k)
    elif op == 'sobel':
        # Apply Sobel on grayscale
        if img.ndim == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            gray = img
        dx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
        dy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
        mag = cv2.magnitude(dx, dy)
        mag = np.clip(mag / (mag.max() + 1e-6) * 255, 0, 255).astype(np.uint8)
        return mag
    elif op == 'canny':
        if img.ndim == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            gray = img
        t1 = int(kwargs.get('t1', 100))
        t2 = int(kwargs.get('t2', 200))
        return cv2.Canny(gray, t1, t2)
    elif op == 'sharpen':
        # Simple unsharp mask style kernel
        k = np.array([[0, -1, 0],
                      [-1, 5, -1],
                      [0, -1, 0]], dtype=np.float32)
        return cv2.filter2D(img, -1, k)
    else:
        raise ValueError(f"Unsupported op: {op}")


# ---------------------------
# Tiling helpers
# ---------------------------

def make_tiles(h: int, w: int, tiles_y: int, tiles_x: int, halo: int) -> List[Tile]:
    tiles: List[Tile] = []
    ys = [round(i * h / tiles_y) for i in range(tiles_y + 1)]
    xs = [round(i * w / tiles_x) for i in range(tiles_x + 1)]

    for j in range(tiles_y):
        for i in range(tiles_x):
            y0, y1 = ys[j], ys[j + 1]
            x0, x1 = xs[i], xs[i + 1]
            # Expand ROI by halo while clamping to image bounds
            ry0 = max(0, y0 - halo)
            rx0 = max(0, x0 - halo)
            ry1 = min(h, y1 + halo)
            rx1 = min(w, x1 + halo)
            # crop region relative to expanded ROI
            cy0 = y0 - ry0
            cx0 = x0 - rx0
            cy1 = cy0 + (y1 - y0)
            cx1 = cx0 + (x1 - x0)
            tiles.append(Tile(rx0, ry0, rx1, ry1, (cx0, cy0, cx1, cy1)))
    return tiles


def _process_tile(args):
    roi, op, kwargs = args
    out = apply_op(roi, op, **kwargs)
    return out


# ---------------------------
# Main parallel runner
# ---------------------------

def run_parallel(img: np.ndarray, op: str, tiles: Tuple[int, int], halo: int, workers: int | None, **kwargs) -> np.ndarray:
    h, w = img.shape[:2]
    tiles_y, tiles_x = tiles
    tile_defs = make_tiles(h, w, tiles_y, tiles_x, halo)

    # Package tasks
    tasks = []
    rois = []
    for t in tile_defs:
        roi = img[t.y0:t.y1, t.x0:t.x1]
        rois.append(roi)
        tasks.append((roi, op, kwargs))

    # Execute in parallel
    start = time.perf_counter()
    with ProcessPoolExecutor(max_workers=workers) as ex:
        futures = [ex.submit(_process_tile, arg) for arg in tasks]
        results = [f.result() for f in futures]
    elapsed = time.perf_counter() - start

    # Create output array with proper shape and dtype
    # Get a sample result to determine output characteristics
    sample_result = results[0]
    if sample_result.ndim == 2:
        out = np.zeros((h, w), dtype=sample_result.dtype)
    else:
        out = np.zeros((h, w, sample_result.shape[2]), dtype=sample_result.dtype)

    # Stitch results back together
    for idx, t in enumerate(tile_defs):
        res = results[idx]
        cx0, cy0, cx1, cy1 = t.crop

        # Extract the region we want from the processed tile
        if res.ndim == 2:
            cropped = res[cy0:cy1, cx0:cx1]
        else:
            cropped = res[cy0:cy1, cx0:cx1, :]

        # Calculate the actual target region in the output
        target_y0 = t.y0
        target_y1 = target_y0 + (cy1 - cy0)
        target_x0 = t.x0
        target_x1 = target_x0 + (cx1 - cx0)

        # Make sure we don't go out of bounds
        target_y1 = min(target_y1, h)
        target_x1 = min(target_x1, w)

        # Adjust cropped region if needed
        actual_h = target_y1 - target_y0
        actual_w = target_x1 - target_x0

        if cropped.ndim == 2:
            cropped = cropped[:actual_h, :actual_w]
            out[target_y0:target_y1, target_x0:target_x1] = cropped
        else:
            cropped = cropped[:actual_h, :actual_w, :]
            out[target_y0:target_y1, target_x0:target_x1, :] = cropped

    return out, elapsed


def run_sequential(img: np.ndarray, op: str, **kwargs) -> Tuple[np.ndarray, float]:
    start = time.perf_counter()
    out = apply_op(img, op, **kwargs)
    elapsed = time.perf_counter() - start
    return out, elapsed


# ---------------------------
# CLI
# ---------------------------

def parse_tiles(s: str) -> Tuple[int, int]:
    if 'x' not in s.lower():
        raise argparse.ArgumentTypeError("Tiles must be in AxB format, e.g., 2x4")
    a, b = s.lower().split('x')
    ty, tx = int(a), int(b)
    if ty < 1 or tx < 1:
        raise argparse.ArgumentTypeError("Tiles must be >= 1x1")
    return ty, tx


def main():
    p = argparse.ArgumentParser(description="Parallel Image Processing (CPU, tile-based)")
    p.add_argument('--input', required=True, help='Path to input image')
    p.add_argument('--output', required=True, help='Path to save output image')
    p.add_argument('--op', default='canny', choices=['gaussian', 'median', 'sobel', 'canny', 'sharpen'])
    p.add_argument('--tiles', type=parse_tiles, default=(2, 2), help='Tiles as AxB, e.g., 2x4 (A=rows, B=cols)')
    p.add_argument('--halo', type=int, default=8, help='Overlap (pixels) added around each tile')
    p.add_argument('--workers', type=int, default=None, help='Number of worker processes (default: os.cpu_count())')
    p.add_argument('--grayscale', action='store_true', help='Force grayscale before processing (except ops that convert internally)')
    # op-specific knobs
    p.add_argument('--ksize', type=int, default=5, help='Kernel size for gaussian/median')
    p.add_argument('--sigma', type=float, default=1.0, help='Sigma for gaussian')
    p.add_argument('--t1', type=int, default=100, help='Canny threshold1')
    p.add_argument('--t2', type=int, default=200, help='Canny threshold2')
    args = p.parse_args()

    if not os.path.isfile(args.input):
        raise SystemExit(f"Input not found: {args.input}")

    img = cv2.imread(args.input, cv2.IMREAD_COLOR)
    if img is None:
        raise SystemExit("Failed to load image. Unsupported format or corrupt file.")

    if args.grayscale and args.op not in ('canny', 'sobel'):
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    print(f"Input image shape: {img.shape}")

    # Sequential reference
    seq_out, t_seq = run_sequential(img, args.op, ksize=args.ksize, sigma=args.sigma, t1=args.t1, t2=args.t2)
    print(f"Sequential output shape: {seq_out.shape}")

    # Parallel
    par_out, t_par = run_parallel(img, args.op, args.tiles, args.halo, args.workers, ksize=args.ksize, sigma=args.sigma, t1=args.t1, t2=args.t2)
    print(f"Parallel output shape: {par_out.shape}")

    # Prefer parallel output to save (identical visually except for floating rounding)
    out = par_out

    # If single-channel, save as PNG to preserve 8-bit grayscale; for multi-channel keep as BGR
    if out.ndim == 2:
        save_ok = cv2.imwrite(args.output, out)
    else:
        save_ok = cv2.imwrite(args.output, out)

    if not save_ok:
        raise SystemExit("Failed to save output image.")

    speedup = t_seq / t_par if t_par > 0 else float('inf')
    print(f"Operation      : {args.op}")
    print(f"Image          : {args.input} -> {args.output}")
    print(f"Tiles (YxX)    : {args.tiles[0]}x{args.tiles[1]} | Halo: {args.halo} | Workers: {args.workers or os.cpu_count()}")
    print(f"Sequential time: {t_seq*1000:.2f} ms")
    print(f"Parallel time  : {t_par*1000:.2f} ms")
    print(f"Speedup        : {speedup:.2f}x")

if __name__ == "__main__":
    import sys
    if len(sys.argv) == 1:  # running in Colab/Notebook with no CLI args
        args = argparse.Namespace(
            input="sample.jpg",
            output="out.jpg",
            op="canny",
            tiles=(2, 2),
            halo=8,
            workers=None,
            grayscale=False,
            ksize=5,
            sigma=1.0,
            t1=100,
            t2=200,
        )
        img = cv2.imread(args.input, cv2.IMREAD_COLOR)
        out, _ = run_parallel(img, args.op, args.tiles, args.halo, args.workers,
                              ksize=args.ksize, sigma=args.sigma, t1=args.t1, t2=args.t2)
        cv2.imwrite(args.output, out)
        print(f"Saved {args.output}")
    else:
        main()

In [ ]:
#!/usr/bin/env python3
"""
Threaded Parallel Image Processing (CPU, tile-based)

Features
- Runs classic filters using ThreadPoolExecutor instead of ProcessPoolExecutor
- Much lower overhead than multiprocessing
- Better for I/O bound and moderate CPU tasks
- Splits image into tiles with halo/overlap so edges are correct

Usage
------
python threaded_image_processing.py \
  --input input.jpg \
  --output out.jpg \
  --op canny \
  --tiles 4x4

Supported ops: gaussian, median, sobel, canny, sharpen
"""

from __future__ import annotations
import argparse
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from typing import List, Tuple
import time

import cv2
import numpy as np


# ---------------------------
# Utility dataclasses
# ---------------------------
@dataclass
class Tile:
    x0: int
    y0: int
    x1: int
    y1: int
    crop: Tuple[int, int, int, int]  # (cx0, cy0, cx1, cy1) relative to ROI


# ---------------------------
# Operations
# ---------------------------

def apply_op(img: np.ndarray, op: str, **kwargs) -> np.ndarray:
    op = op.lower()
    if op == 'gaussian':
        k = int(kwargs.get('ksize', 5))
        k = k if k % 2 == 1 else k + 1
        sigma = float(kwargs.get('sigma', 1.0))
        return cv2.GaussianBlur(img, (k, k), sigma)
    elif op == 'median':
        k = int(kwargs.get('ksize', 5))
        k = k if k % 2 == 1 else k + 1
        return cv2.medianBlur(img, k)
    elif op == 'sobel':
        if img.ndim == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            gray = img
        dx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
        dy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
        mag = cv2.magnitude(dx, dy)
        mag = np.clip(mag / (mag.max() + 1e-6) * 255, 0, 255).astype(np.uint8)
        return mag
    elif op == 'canny':
        if img.ndim == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            gray = img
        t1 = int(kwargs.get('t1', 100))
        t2 = int(kwargs.get('t2', 200))
        return cv2.Canny(gray, t1, t2)
    elif op == 'sharpen':
        k = np.array([[0, -1, 0],
                      [-1, 5, -1],
                      [0, -1, 0]], dtype=np.float32)
        return cv2.filter2D(img, -1, k)
    else:
        raise ValueError(f"Unsupported op: {op}")


# ---------------------------
# Tiling helpers
# ---------------------------

def make_tiles(h: int, w: int, tiles_y: int, tiles_x: int, halo: int) -> List[Tile]:
    tiles: List[Tile] = []
    ys = [round(i * h / tiles_y) for i in range(tiles_y + 1)]
    xs = [round(i * w / tiles_x) for i in range(tiles_x + 1)]

    for j in range(tiles_y):
        for i in range(tiles_x):
            y0, y1 = ys[j], ys[j + 1]
            x0, x1 = xs[i], xs[i + 1]
            # Expand ROI by halo while clamping to image bounds
            ry0 = max(0, y0 - halo)
            rx0 = max(0, x0 - halo)
            ry1 = min(h, y1 + halo)
            rx1 = min(w, x1 + halo)
            # crop region relative to expanded ROI
            cy0 = y0 - ry0
            cx0 = x0 - rx0
            cy1 = cy0 + (y1 - y0)
            cx1 = cx0 + (x1 - x0)
            tiles.append(Tile(rx0, ry0, rx1, ry1, (cx0, cy0, cx1, cy1)))
    return tiles


class TileProcessor:
    """Wrapper to avoid pickling issues and share data efficiently"""
    def __init__(self, img: np.ndarray, op: str, kwargs: dict):
        self.img = img
        self.op = op
        self.kwargs = kwargs

    def process_tile(self, tile: Tile) -> Tuple[np.ndarray, Tile]:
        # Extract ROI
        roi = self.img[tile.y0:tile.y1, tile.x0:tile.x1]
        # Process it
        result = apply_op(roi, self.op, **self.kwargs)
        return result, tile


# ---------------------------
# Main parallel runner
# ---------------------------

def run_threaded_parallel(img: np.ndarray, op: str, tiles: Tuple[int, int], halo: int, workers: int | None, **kwargs) -> Tuple[np.ndarray, float]:
    h, w = img.shape[:2]
    tiles_y, tiles_x = tiles
    tile_defs = make_tiles(h, w, tiles_y, tiles_x, halo)

    # Create processor with shared data
    processor = TileProcessor(img, op, kwargs)

    # Execute in parallel using threads
    start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=workers) as executor:
        # Submit all tasks
        futures = [executor.submit(processor.process_tile, tile) for tile in tile_defs]
        # Collect results
        results = []
        for future in as_completed(futures):
            result, tile = future.result()
            results.append((result, tile))
    elapsed = time.perf_counter() - start

    # Sort results by original tile order to maintain consistency
    tile_to_index = {id(tile): i for i, tile in enumerate(tile_defs)}
    results.sort(key=lambda x: tile_to_index[id(x[1])])

    # Create output array with proper shape and dtype
    sample_result = results[0][0]
    if sample_result.ndim == 2:
        out = np.zeros((h, w), dtype=sample_result.dtype)
    else:
        out = np.zeros((h, w, sample_result.shape[2]), dtype=sample_result.dtype)

    # Stitch results back together
    for result, tile in results:
        cx0, cy0, cx1, cy1 = tile.crop

        # Extract the region we want from the processed tile
        if result.ndim == 2:
            cropped = result[cy0:cy1, cx0:cx1]
        else:
            cropped = result[cy0:cy1, cx0:cx1, :]

        # Calculate target region in output
        target_y0 = tile.y0
        target_y1 = target_y0 + (cy1 - cy0)
        target_x0 = tile.x0
        target_x1 = target_x0 + (cx1 - cx0)

        # Bounds checking
        target_y1 = min(target_y1, h)
        target_x1 = min(target_x1, w)

        # Adjust cropped region if needed
        actual_h = target_y1 - target_y0
        actual_w = target_x1 - target_x0

        if cropped.ndim == 2:
            cropped = cropped[:actual_h, :actual_w]
            out[target_y0:target_y1, target_x0:target_x1] = cropped
        else:
            cropped = cropped[:actual_h, :actual_w, :]
            out[target_y0:target_y1, target_x0:target_x1, :] = cropped

    return out, elapsed


def run_sequential(img: np.ndarray, op: str, **kwargs) -> Tuple[np.ndarray, float]:
    start = time.perf_counter()
    out = apply_op(img, op, **kwargs)
    elapsed = time.perf_counter() - start
    return out, elapsed


# ---------------------------
# CLI
# ---------------------------

def parse_tiles(s: str) -> Tuple[int, int]:
    if 'x' not in s.lower():
        raise argparse.ArgumentTypeError("Tiles must be in AxB format, e.g., 2x4")
    a, b = s.lower().split('x')
    ty, tx = int(a), int(b)
    if ty < 1 or tx < 1:
        raise argparse.ArgumentTypeError("Tiles must be >= 1x1")
    return ty, tx


def main():
    p = argparse.ArgumentParser(description="Threaded Parallel Image Processing")
    p.add_argument('--input', required=True, help='Path to input image')
    p.add_argument('--output', required=True, help='Path to save output image')
    p.add_argument('--op', default='canny', choices=['gaussian', 'median', 'sobel', 'canny', 'sharpen'])
    p.add_argument('--tiles', type=parse_tiles, default=(2, 2), help='Tiles as AxB, e.g., 2x4')
    p.add_argument('--halo', type=int, default=8, help='Overlap pixels around each tile')
    p.add_argument('--workers', type=int, default=None, help='Number of worker threads (default: 4)')
    p.add_argument('--grayscale', action='store_true', help='Force grayscale before processing')
    # op-specific parameters
    p.add_argument('--ksize', type=int, default=5, help='Kernel size for gaussian/median')
    p.add_argument('--sigma', type=float, default=1.0, help='Sigma for gaussian')
    p.add_argument('--t1', type=int, default=100, help='Canny threshold1')
    p.add_argument('--t2', type=int, default=200, help='Canny threshold2')

    args = p.parse_args()

    if not os.path.isfile(args.input):
        raise SystemExit(f"Input not found: {args.input}")

    img = cv2.imread(args.input, cv2.IMREAD_COLOR)
    if img is None:
        raise SystemExit("Failed to load image. Unsupported format or corrupt file.")

    if args.grayscale and args.op not in ('canny', 'sobel'):
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    print(f"Input image shape: {img.shape}")

    # Set default workers for threading (usually 4 is good for I/O bound tasks)
    if args.workers is None:
        args.workers = 4

    # Sequential reference
    seq_out, t_seq = run_sequential(img, args.op, ksize=args.ksize, sigma=args.sigma, t1=args.t1, t2=args.t2)
    print(f"Sequential output shape: {seq_out.shape}")

    # Threaded parallel
    par_out, t_par = run_threaded_parallel(img, args.op, args.tiles, args.halo, args.workers,
                                         ksize=args.ksize, sigma=args.sigma, t1=args.t1, t2=args.t2)
    print(f"Threaded parallel output shape: {par_out.shape}")

    # Save parallel output
    out = par_out
    save_ok = cv2.imwrite(args.output, out)
    if not save_ok:
        raise SystemExit("Failed to save output image.")

    speedup = t_seq / t_par if t_par > 0 else float('inf')
    print(f"Operation         : {args.op}")
    print(f"Image             : {args.input} -> {args.output}")
    print(f"Tiles (YxX)       : {args.tiles[0]}x{args.tiles[1]} | Halo: {args.halo} | Workers: {args.workers}")
    print(f"Sequential time   : {t_seq*1000:.2f} ms")
    print(f"Threaded time     : {t_par*1000:.2f} ms")
    print(f"Speedup           : {speedup:.2f}x")
    print(f"Threading overhead: {((t_par - t_seq)/t_seq)*100:+.1f}%")


if __name__ == "__main__":
    main()